# DataFest -  Overview

This notebook showcases a dataset of Wikipedia edits for analysis.

- Ten large Wikipedia language editions: English (`enwiki`), German (`dewiki`), French (`frwiki`), Swedish (`svwiki`), Dutch (`nlwiki`), Russian (`ruwiki`), Spanish (`eswiki`), Italian (`itwiki`), Arabic (`arwiki`), and Polish (`plwiki`).
- Top 10,000 most-read articles in each of those language editions from all of 2025.

There are three kinds of information: 
- page level (information about the page itself)
- page view (how often is the page viewed) and
- page edits (what were the edits made throughout the year).

First the download links and cache functions are declared:

In [21]:
import polars as pl
import json
import urllib.request
from pathlib import Path

def cached(url):
    """Download a file from a URL and cache it locally. Returns the local path, skipping download if already cached."""
    path = Path(url.split("/")[-1])
    if not path.exists():
        print(f"Downloading {path.name}...")
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp, open(path, "wb") as f:
            f.write(resp.read())
        print(f"Downloaded {path.name}")
    return path

def get_page_edits(language):
    return f"https://analytics.wikimedia.org/published/datasets/one-off/datafete/edit_types/{language}wiki.json.gz"

all_languages = ["ar", "de", "en", "es", "fr", "it", "nl", "pl", "ru", "sv"]

page_data_fn = "https://analytics.wikimedia.org/published/datasets/one-off/datafete/page_info.json.gz"  # 166MB
page_views_fn = "https://analytics.wikimedia.org/published/datasets/one-off/datafete/page_views.json.gz"  # 319MB
page_edits_de_fn = get_page_edits("de")  # 138MB


# Page Level Information

- `wiki_db`: which language version (e.g. `svwiki`)
- `page_id`: for a stable identifier (e.g., `https:<lang>.wikipedia.org/wiki/?curid={page_id}`)
- `qid` as a language-agnostic identifier (in Wikidata) to indicate when articles in separate language editions are about the same topic (`www.wikidata.org/wiki/{qid}`)
- `embedding`: A 50-dimensional topic embedding based on the page's links. These are language-agnostic so comparable across language editions. See [model details](https://meta.wikimedia.org/wiki/Machine_learning_models/Production/Language_agnostic_link-based_article_topic)
- `predicted_labels`: Predicted high-level topics for the article from the same model as above
- `pageviews`: Total number of pageviews in 2025
- `page_title`: the title of the page (for easy interpretability)

In [22]:
page_data = pl.read_ndjson(cached(page_data_fn)) # , n_rows=10000)
print(f"length of page_data: {len(page_data)}")
page_data.head()

length of page_data: 99732


wiki_db,page_id,qid,embedding,predicted_labels,pageviews,page_title
str,i64,str,list[f64],list[struct[2]],i64,str
"""svwiki""",78541,"""Q847769""","[-0.281103, 0.192694, … -0.009869]","[{""Geography.Regions.Europe.Northern_Europe"",1.00001}, {""Geography.Regions.Europe.Europe*"",0.9988405}, … {""Culture.Biography.Biography*"",0.00001}]",21556,"""Filipstad"""
"""svwiki""",12392,"""Q764""","[0.107437, -0.233238, … 0.031688]","[{""STEM.STEM*"",1.00001}, {""STEM.Biology"",0.9987652}, … {""Culture.Media.Music"",0.00001}]",19754,"""Svampar"""
"""svwiki""",34040,"""Q17293""","[-0.147307, 0.18542, … 0.002733]","[{""Culture.Biography.Biography*"",0.930468}, {""Geography.Regions.Asia.Asia*"",0.55448}, … {""Culture.Performing_arts"",0.00001}]",18549,"""Tenzin_Gyatso"""
"""svwiki""",139198,"""Q4939352""","[-0.166814, -0.113468, … -0.097478]","[{""Geography.Regions.Europe.Northern_Europe"",0.990884}, {""Geography.Regions.Europe.Europe*"",0.980886}, … {""STEM.Space"",0.00001}]",60279,"""Jannike_Björling"""
"""svwiki""",55797,"""Q6210414""","[-0.13229, 0.008684, … -0.150917]","[{""Culture.Biography.Biography*"",0.9978273}, {""Culture.Media.Music"",0.9802909}, … {""History_and_Society.Transportation"",0.00001}]",45725,"""Owe_Thörnqvist"""


# Page View

- `wiki_db`: which language version (e.g. `svwiki`)
- `page_id`: for a stable identifier (e.g., `https:<lang>.wikipedia.org/wiki/?curid={page_id}`)
- `day`: the actual day formatted as `YYYY-MM-DD`
- `pageviews`: the number of page views for this day

In [23]:
page_views = pl.read_ndjson(cached(page_views_fn)) # , n_rows=10000)
print(f"length of page_views: {len(page_views)}")
page_views.head()

length of page_views: 36249509


wiki_db,page_id,day,pageviews
str,i64,str,i64
"""dewiki""",2365952,"""2025-09-26""",339
"""eswiki""",4715,"""2025-09-20""",2201
"""enwiki""",11715,"""2025-09-23""",4023
"""ruwiki""",18097,"""2025-09-25""",1047
"""frwiki""",7226,"""2025-09-25""",365


# Page Edit Information

Information about all edits to those articles in 2025.
Separate files for each language edition. Files can be rather large (`de` has 138 MB whereas `en` has 1.2 GB ).

- `revision_id`: unique identifier for each edit/revision
- `page_id`: for a stable identifier linking back to the page (e.g., `https:<lang>.wikipedia.org/wiki/?curid={page_id}`)
- `user_text`: the username of the user who made the edit
- `revision_timestamp`: timestamp of when the edit was made (formatted as ISO 8601, e.g. `2025-01-02T09:52:22.000Z`)
- `revision_comment`: comment (if any) the editor added to describe the edit
- `edit_types_json`: computed "edit types" that indicate what was changed by the edit -- see [mwedittypes](https://pypi.org/project/mwedittypes/) library for more details (further details below)
- `revision_tags`: revision tags that are automatically applied. Descriptions can be found at corresponding `https://<lang>.wikipedia.org/wiki/Special:Tags`. Most useful are probably those related to whether an edit was [reverted](https://en.wikipedia.org/wiki/Wikipedia:Reverting) (`mw-reverted`) or itself was reverting another edit (presence of either `mw-undo`, `mw-rollback`, or `mw-manual-revert`)
- `is_bot`: boolean indicating whether the edit was made by a bot account



In [24]:
page_edits_de = pl.read_ndjson(cached(page_edits_de_fn)) # , n_rows=10000)
print(f"length of page_edits_de: {len(page_edits_de)}")
page_edits_de.head()

length of page_edits_de: 349281


revision_id,page_id,user_text,revision_timestamp,revision_comment,edit_types_json,revision_tags,is_bot
i64,i64,str,str,str,str,list[str],bool
251831096,8198,"""Dragonlord73""","""2025-01-02T09:52:22.000Z""","""Korrektur""","""{""node-edits"": [[""Table"", ""cha…","[""wikieditor""]",false
251776145,999397,"""Gmünder""","""2025-01-01T07:08:20.000Z""","""akt""","""{""node-edits"": [[""Wikilink"", ""…","[""wikieditor""]",false
251839625,18194,"""EinBeitrag""","""2025-01-02T14:47:13.000Z""","""Literatur ergänzt, zugeordnet""","""{""node-edits"": [[""List"", ""remo…","[""wikieditor""]",false
251807171,8528048,"""Kito9999""","""2025-01-01T14:45:26.000Z""","""Die letzte Textänderung von [[…","""{""node-edits"": [[""Table"", ""cha…","[""mw-manual-revert""]",false
251839733,869243,"""Scholle2008""","""2025-01-02T14:51:13.000Z""","""/* Erfolgreichste Filme */ Akt…","""{""node-edits"": [[""Template"", ""…","[""visualeditor""]",false


### Details for `edit_types_json`

In [25]:
for revision in page_edits_de.limit(5).to_dicts():
    et = json.loads(revision["edit_types_json"])
    if et["node-edits"]:
        print(f"revision {revision['revision_id']} page {revision['page_id']}")
        for ne in et["node-edits"]:
            print(f"\t{ne}")

revision 251831096 page 8198
	['Table', 'change', '9: === Größte Städte ===', None, [['cells', 'change', 21.0]]]
	['Table', 'change', '9: === Größte Städte ===', None, [['cells', 'change', 20.0]]]
	['Text Formatting', 'change', '9: === Größte Städte ===', None, []]
	['Wikilink', 'move', '9: === Größte Städte ===', 'North Lanarkshire', []]
	['Wikilink', 'move', '9: === Größte Städte ===', 'Airdrie (North Lanarkshire)', []]
	['Wikilink', 'move', '9: === Größte Städte ===', 'North Lanarkshire', []]
	['Wikilink', 'move', '9: === Größte Städte ===', 'Coatbridge', []]
	['Wikilink', 'move', '9: === Größte Städte ===', 'Dunfermline', []]
	['Wikilink', 'move', '9: === Größte Städte ===', 'Fife (Schottland)', []]
	['Wikilink', 'move', '9: === Größte Städte ===', 'East Ayrshire', []]
	['Wikilink', 'move', '9: === Größte Städte ===', 'Kilmarnock', []]
revision 251776145 page 999397
	['Wikilink', 'insert', '55: === Behörden ===', 'Oberfinanzdirektion Baden-Württemberg', [['title', None, 'Oberfinanz